In [ ]:
import matplotlib.pyplot as plt
import sys
import os
import importlib
import pandas as pd
import xarray as xr

# Add the parent directory to the path to import src as a package
sys.path.insert(0, os.path.abspath('..'))
from src import dataloader

importlib.reload(dataloader)
from src import utils
from src import export

importlib.reload(export)
from src.export import write_dyad_to_uniwaw_imported

%matplotlib inline 
plot_flag = True

# Example of saving data for multiple dyads to NCDF while creating the folder structure

In [ ]:
input_folder = "/Users/admin/Library/CloudStorage/GoogleDrive-j.zygierewicz@uw.edu.pl/.shortcut-targets-by-id/1N4ySQ5GO6UE8fY2jnRkRUjBFm4XHrBRv/SYNCC-IN/WP4          - Joint study/UniWAW Data collection/UNIWAW_RAW_DATA"
export_folder = "/Users/admin/Library/CloudStorage/GoogleDrive-j.zygierewicz@uw.edu.pl/.shortcut-targets-by-id/1N4ySQ5GO6UE8fY2jnRkRUjBFm4XHrBRv/SYNCC-IN/WP4          - Joint study/UniWAW Data collection/UNIWAW_EEG_exported"

In [ ]:
# create a list of dyads to export
dyades_to_export = ['W_009']
# load the metadata file to get the list of all dyads and their corresponding movie durations
metadata_file = os.path.join(input_folder, "meta_data.csv")
metadata_df = pd.read_csv(metadata_file, sep=';')
# filter the dyades_to_export list to include only the dyads in the metadata file
dyades_to_export = [dyad for dyad in dyades_to_export if dyad in metadata_df['ID'].values]
dyades_to_export = sorted(dyades_to_export)
print(f"Exporting {len(dyades_to_export)} dyads: {dyades_to_export}")

In [ ]:
metadata_df=metadata_df[metadata_df['ID'].isin(dyades_to_export)]
metadata_df=metadata_df.sort_values(by='ID')
metadata_df.reset_index(drop=True, inplace=True)
print(metadata_df)

In [ ]:
failed_dyads = []
for dyad in dyades_to_export:
    try:
        print(f"Exporting {dyad}...")
        write_dyad_to_uniwaw_imported(
            [dyad],
            input_data_path=input_folder,
            export_path=export_folder,
            load_et=True,
            eeg_filter_type='fir',
            time_margin=10,
            verbose=True,
            plot_flag=True
        )
        print(f"Done: {dyad}")
    except Exception as e:
        failed_dyads.append((dyad, str(e)))
        print(f"Failed: {dyad} -> {e}")

print(f"Finished. Success: {len(dyades_to_export) - len(failed_dyads)}, Failed: {len(failed_dyads)}")
if failed_dyads:
    print("Failed dyads:")
    for dyad, err in failed_dyads:
        print(f"  - {dyad}: {err}")

# Example of reading from ncdf file to xarray

> Note: batch export now also writes RMSSD modality files under `RMSSD/<dyad_id>/<member>/...` when RMSSD is available in the loaded multimodal data.